# 04 — DeFi Merge: Liquidations → Econometric Panel (Report)

Reading-side companion to `run_defi_merge.py`. The script merges the
hourly DeFi liquidation feed onto the pre-DeFi core panel and dumps
three artefacts:

- `data/econ/econ_core_full_1h.parquet` — the 27-column, 41 328-row panel
- `data/analysis/reports/defi_merge_qa.json` — merge audit
- `data/analysis/reports/stationarity_adf.json` — ADF unit-root tests

*(Note: the 4-column side artefact `liquidations_1h.parquet` was deprecated in v2.x — see `run_defi_merge.py` CHANGELOG.)*

This notebook reads those files and verifies the panel against the
downstream contract without re-running any computation. This report
supersedes the original exploratory notebook (`04_defi_merge.ipynb`),
which is not part of the replication package.

**Contract with NB07 / NB08 / NB09.** `econ_core_full_1h.parquet`
schema is locked: 27 columns in this exact order, with `n_liquidations`
upcast to float64 (preserved verbatim — int→float drift would break
consumers). Any drift breaks every downstream notebook + factorised
script.

## 0. Setup & auto-execution

In [ ]:
# ── Setup ──
import sys, json, subprocess, time
from pathlib import Path
sys.path.insert(0, "..")
from config import CFG, ECON_DIR, REPORTS_DIR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Toggles
FORCE_RERUN = False    # if True, regenerate all outputs via the script

SCRIPT  = Path.cwd().parent / "scripts" / "run_defi_merge.py"
OUTPUTS = {
    "panel":     ECON_DIR    / "econ_core_full_1h.parquet",
    "qa":        REPORTS_DIR / "defi_merge_qa.json",
    "adf":       REPORTS_DIR / "stationarity_adf.json",
}

print("Setup OK")
print(f"  FORCE_RERUN = {FORCE_RERUN}")

In [ ]:
# ── Auto-execute if outputs are missing or FORCE_RERUN is set ──
missing = [t for t, p in OUTPUTS.items() if not p.exists()]

if FORCE_RERUN:
    print("FORCE_RERUN=True → re-running run_defi_merge.py...", flush=True)
    run = True
elif missing:
    print(f"Outputs missing ({missing}) → running run_defi_merge.py...", flush=True)
    run = True
else:
    print("All outputs present, skipping recompute (FORCE_RERUN=False)")
    for t, p in OUTPUTS.items():
        print(f"  [{t}] {p.name}")
    run = False

if run:
    assert SCRIPT.exists(), SCRIPT
    cmd = [sys.executable, str(SCRIPT)]
    print("Running:", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, cwd=str(Path.cwd().parent), check=True)
    print(f"\nScript done — {time.time()-t0:.1f}s")

In [ ]:
# ── Load artefacts ──
df = pd.read_parquet(OUTPUTS["panel"], engine="pyarrow")
df["date"] = pd.to_datetime(df["date"], utc=True)
with open(OUTPUTS["qa"]) as f:
    qa = json.load(f)
with open(OUTPUTS["adf"]) as f:
    adf = json.load(f)

print(f"panel: {len(df):,} rows × {df.shape[1]} cols")
print(f"qa:    status = {qa['status']}")
print(f"adf:   {len(adf)} series tested")

## 1. Description du panel produit

Bornes temporelles, taille du fichier, cadence horaire. Doit afficher
41 328 lignes × 27 colonnes sur `[2021-03-15, 2025-12-01)` UTC.

In [ ]:
print("═══ Description du panel ═══\n")
size_mb = OUTPUTS["panel"].stat().st_size / 1e6
print(f"  Rows:       {len(df):,}")
print(f"  Cols:       {df.shape[1]}")
print(f"  Date range: [{df['date'].min()}, {df['date'].max()}]")
print(f"  Frequency:  {CFG.FREQ}")
print(f"  File size:  {size_mb:.2f} MB")

## 2. Statistiques des colonnes DeFi

Distributions des nouvelles colonnes ajoutées par NB04. `liq_usd_total`
en USD bruts, `log_liq = ln(1 + L_t)` pour la régression. ~27 % des
heures contiennent au moins une liquidation enregistrée ; le régime
de stress (top 5 % des heures non-zéro) couvre ~1.3 % du panel.

In [ ]:
defi_cols = ["liq_usd_total", "liq_usd_collateral", "n_liquidations",
             "log_liq", "log_liq_lag1", "shock_x_oi"]
stats = df[defi_cols].describe().T.round(4)
print("═══ Summary stats — colonnes DeFi ═══\n")
print(stats.to_string())

print("\n── Couverture ──")
n_with_liq = int((df["liq_usd_total"] > 0).sum())
print(f"  Heures avec liquidation : {n_with_liq:,} ({100*n_with_liq/len(df):.1f}%)")
print(f"  Heures à zéro           : {len(df)-n_with_liq:,} ({100*(len(df)-n_with_liq)/len(df):.1f}%)")

print("\n── Régime de stress (liq_stress) ──")
print(df["liq_stress"].value_counts().sort_index().to_string())
print(f"  Seuil (USD)             : ${qa['stress_threshold_usd']:,.0f}")

## 3. Validation des fingerprints

Trois sommes et un dtype suffisent à attraper toute régression de la
chaîne build (NB03 + NB04). Si l'un de ces asserts échoue, recomparer
bit-à-bit avec une exécution antérieure connue via
`pd.testing.assert_frame_equal(check_exact=True, check_dtype=True)`.

In [ ]:
print("═══ Fingerprint validation ═══\n")
checks = [
    ("liq_usd_total.sum()", df["liq_usd_total"].sum(), 2_521_318_539.578, 1e-3),
    ("log_liq.sum()",       df["log_liq"].sum(),       71_233.7838,        1e-3),
    ("liq_stress.sum()",    int(df["liq_stress"].sum()), 549,              0),
]
for name, got, expected, tol in checks:
    ok = abs(got - expected) <= tol if tol else got == expected
    flag = "OK" if ok else "FAIL"
    print(f"  [{flag}] {name:24s}: {got!r}  (expected {expected!r})")
    assert ok, name

dtype_ok = df["n_liquidations"].dtype == "float64"
print(f"  [{'OK' if dtype_ok else 'FAIL'}] n_liquidations.dtype     : "
      f"{df['n_liquidations'].dtype}  (expected float64)")
assert dtype_ok, "n_liquidations dtype regression"

## 4. Cross-check événements connus

Sanité de l'alignement temporel : 6 dates de crash bien documentées
doivent montrer un pic de liquidation et un rendement ETH moyen négatif.
Repris verbatim de la cellule `e93fef6e` du notebook original.

In [ ]:
events = {
    "May 19 2021 crash":           ("2021-05-19", "2021-05-20"),
    "Dec 4 2021 flash crash":      ("2021-12-03", "2021-12-05"),
    "Jun 2022 Terra/Luna":         ("2022-06-12", "2022-06-15"),
    "Nov 2022 FTX":                ("2022-11-08", "2022-11-11"),
    "Aug 5 2024 yen carry unwind": ("2024-08-05", "2024-08-06"),
    "Feb 3 2025 DeepSeek":         ("2025-02-02", "2025-02-04"),
}

print("═══ Known Event Cross-Check ═══\n")
for name, (start, end) in events.items():
    mask = (df["date"] >= start) & (df["date"] < end)
    sub = df[mask]
    total_liq = sub["liq_usd_total"].sum()
    peak_liq  = sub["liq_usd_total"].max()
    mean_ret  = sub["ret_eth_perp"].mean()
    print(f"{name:35s}: liq=${total_liq/1e6:>7.1f}M  peak=${peak_liq/1e6:>7.1f}M/h  "
          f"avg_ret={mean_ret:>+.3f}%/h")

## 5. Inventory final

Récapitulatif structurel des 27 colonnes du panel, regroupées par
fonction. Doit lister les 7 colonnes ajoutées par NB04 dans la
section *DeFi liquidations*. Adapté de la cellule `91a474ef`
du notebook original.

In [ ]:
col_groups = {
    "Identifiers":       ["date"],
    "Prices & returns":  ["close_perp", "ret_eth_perp", "close_btc_spot",
                          "ret_btc_spot", "close_eth_spot", "ret_eth_spot"],
    "Leverage proxies":  ["oi", "d_oi", "oi_zscore", "oi_high", "funding_high",
                          "oi_vol_ratio", "funding_rate", "basis_bps"],
    "Volatility":        ["vol_eth_7d", "vol_btc_7d",
                          "ret_eth_std", "ret_btc_std"],
    "DeFi liquidations": ["liq_usd_total", "liq_usd_collateral",
                          "n_liquidations", "log_liq", "log_liq_lag1",
                          "liq_stress", "shock_x_oi"],
    "Volume":            ["volume_perp"],
}
print("═══ Final Panel Columns ═══\n")
for group, cols in col_groups.items():
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    line = f"  {group:20s}: {len(present)} present"
    if missing:
        line += f", {len(missing)} missing: {missing}"
    print(line)
print(f"\nTotal: {df.shape[1]} columns")

## 6. Sanity check graphique — `log_liq` au cours du temps

`log_liq` est le canari de la chaîne complète NB03 + NB04. Doit
afficher une activité bruitée mais non vide, avec des pics nets sur
les événements de crash répertoriés en §4 (mai-2021, juin-2022,
août-2024, févr-2025). Toute absence de signal sur ces dates est
suspecte (problème d'alignement temporel ou d'extraction Dune).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))
ax.plot(df["date"], df["log_liq"], color="black", linewidth=0.4)
ax.set_xlabel("")
ax.set_ylabel("log_liq")
ax.margins(x=0)
plt.tight_layout()
plt.show()

## Next step

`econ_core_full_1h.parquet` est le panel terminal de la chaîne
builder. Il est consommé par :
- `scripts/paper/make_figures.py` (figures du papier)
- **NB07** / `run_quantile_lp.py` (quantile local projections)
- **NB08** / `run_robustness_all.py` (robustness checks)
- **NB09** (descriptive stats)
- `run_bootstrap.py`

Toute régression sur ce parquet est immédiatement visible en aval.